# Notebook 5: Leave-One-Out Cross-Validation (LOOCV)

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 1.5 hours  
**Week 7 — Model Evaluation, Validation & Metrics**

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain what LOOCV is and how it differs from k-fold CV
2. Implement LOOCV from scratch using a loop
3. Compute LOOCV analytically for linear models using the hat matrix
4. Identify when LOOCV is the right choice — and when to avoid it
5. Compare the bias-variance tradeoff across CV strategies

---

## Why Does This Matter?

In fraud detection, labeled data is expensive and rare. You might have only 40–60 confirmed fraud cases to train on. Holding out 20% for validation wastes 8–12 precious examples. LOOCV lets you train on **all but one** sample at a time, squeezing every drop of signal from a small dataset.

But LOOCV is not free — it costs O(n) model fits. On a dataset of 10,000 samples, that is 10,000 full training runs. Knowing *when* to pay that cost — and when to use 5-fold instead — is a practical skill every ML practitioner needs.

## Real-World Analogy: The Oral Exam Board

Imagine a bank is certifying 100 fraud analysts. The certification board wants to assess each analyst as objectively as possible.

**Method A — LOOCV Style:**  
For each analyst, the other 99 analysts collaborate to design the hardest, most representative exam possible. That one analyst sits the exam alone. Then you move on to the next analyst. You get 100 exam scores, one per person, each based on a test designed by the other 99. The assessment is extremely thorough and barely wastes anyone's knowledge — but running 100 separate exam sessions is exhausting.

**Method B — 5-Fold Style:**  
Split the 100 analysts into 5 groups of 20. Each group takes a test designed by the other 80. Faster, but each test is designed by fewer people and each group of 20 is a rougher approximation of the full cohort.

**The LOOCV insight:** When n is small, every data point is precious. LOOCV is the "leave no knowledge on the table" approach to validation.

---

## Concept in Plain English

**Leave-One-Out Cross-Validation** is just k-fold CV where k = n (the number of training samples).

For each sample i from 1 to n:
1. **Remove** sample i from the dataset
2. **Train** the model on the remaining n-1 samples
3. **Predict** on sample i
4. **Record** the error on that one prediction

Final score = average of all n individual errors.

$$\text{LOOCV Score} = \frac{1}{n} \sum_{i=1}^{n} L(y_i, \hat{y}_i^{(-i)})$$

Where $\hat{y}_i^{(-i)}$ means "prediction for sample i, made by a model trained on everything EXCEPT sample i".

---

## The Analytical Shortcut for Linear Models

For Ordinary Least Squares (OLS) regression, you never need to fit n models. The **hat matrix** shortcut lets you compute the LOOCV MSE in a single matrix operation:

$$\text{LOOCV MSE} = \frac{1}{n} \sum_{i=1}^{n} \left( \frac{y_i - \hat{y}_i}{1 - h_{ii}} \right)^2$$

Where:
- $\hat{y}_i$ is the prediction from the **full** model (trained on all n samples)
- $h_{ii}$ is the **leverage** of point i — the i-th diagonal of the hat matrix $H = X(X^\top X)^{-1}X^\top$
- High leverage points (unusual X values) are up-weighted — they matter more to LOOCV

This shortcut does NOT generalise to neural networks or tree-based models because those are not linear in their parameters.

In [ ]:
# ── Cell 3: Imports and global configuration ──────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import time
import warnings

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.model_selection import (
    LeaveOneOut, KFold, cross_val_score, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, mean_squared_error

warnings.filterwarnings('ignore')
np.random.seed(42)

# Plotting style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

print("Libraries loaded successfully.")

In [ ]:
# ── Cell 4: Create small fraud dataset (n=150, ~5% fraud) ────────────────────
# Small dataset — exactly the scenario where LOOCV shines
# Features: transaction_amount, time_of_day, merchant_risk_score,
#           num_transactions_today, distance_from_home

np.random.seed(42)
N = 150
fraud_rate = 0.05
n_fraud = int(N * fraud_rate)        # 7-8 fraud cases
n_legit = N - n_fraud

# Legitimate transactions
legit = pd.DataFrame({
    'transaction_amount': np.random.lognormal(4.0, 0.8, n_legit),
    'time_of_day':        np.random.uniform(6, 22, n_legit),
    'merchant_risk':      np.random.beta(2, 8, n_legit),
    'txn_today':          np.random.poisson(3, n_legit).astype(float),
    'dist_from_home':     np.random.exponential(15, n_legit),
    'is_fraud':           0
})

# Fraud transactions — higher amounts, odd hours, risky merchants
fraud = pd.DataFrame({
    'transaction_amount': np.random.lognormal(5.5, 1.0, n_fraud),
    'time_of_day':        np.random.choice(
                              np.concatenate([np.arange(0, 5), np.arange(23, 25)]),
                              n_fraud).astype(float) % 24,
    'merchant_risk':      np.random.beta(8, 2, n_fraud),
    'txn_today':          np.random.poisson(9, n_fraud).astype(float),
    'dist_from_home':     np.random.exponential(80, n_fraud),
    'is_fraud':           1
})

df = pd.concat([legit, fraud], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

X = df.drop('is_fraud', axis=1).values
y = df['is_fraud'].values

print(f"Dataset shape: {X.shape}")
print(f"Fraud cases:   {y.sum()} ({y.mean()*100:.1f}%)")
print(f"Legit cases:   {(1-y).sum()} ({(1-y).mean()*100:.1f}%)")
print("\nNote: With only", y.sum(), "fraud cases, every sample is precious!")
df.head()

In [ ]:
# ── Cell 5: LOOCV implemented from scratch ────────────────────────────────────
# Manual loop: for each sample i, train on everything else, predict on i

def loocv_from_scratch(X, y, model_class, **model_kwargs):
    """
    Implements LOOCV manually.
    Returns array of per-sample scores (1=correct, 0=wrong for classification).
    """
    n = len(y)
    scores = np.zeros(n)
    scaler = StandardScaler()

    for i in range(n):
        # Build train set: all indices except i
        train_idx = np.concatenate([np.arange(0, i), np.arange(i+1, n)])
        X_train, y_train = X[train_idx], y[train_idx]
        X_test,  y_test  = X[[i]],       y[[i]]

        # Scale using only training data statistics (no leakage!)
        X_train_s = scaler.fit_transform(X_train)
        X_test_s  = scaler.transform(X_test)

        # Fit model and predict
        model = model_class(**model_kwargs)
        model.fit(X_train_s, y_train)
        pred = model.predict(X_test_s)
        scores[i] = (pred[0] == y_test[0])   # 1 if correct

    return scores


# Run LOOCV from scratch
print("Running LOOCV from scratch (150 model fits)...")
t0 = time.time()
loocv_scores_scratch = loocv_from_scratch(
    X, y, LogisticRegression, max_iter=300, random_state=42
)
t_loocv_scratch = time.time() - t0

print(f"  LOOCV Accuracy (scratch): {loocv_scores_scratch.mean():.4f}")
print(f"  Time taken:               {t_loocv_scratch:.2f}s")
print(f"  Predictions per fold:     1  (n_test = 1 always)")

In [ ]:
# ── Cell 6: sklearn LOOCV vs k-fold — timing comparison ──────────────────────
# Show the practical cost difference between CV strategies on n=150

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=300, random_state=42))
])

cv_configs = {
    'LOOCV (k=150)':  LeaveOneOut(),
    '10-Fold CV':     StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
    '5-Fold CV':      StratifiedKFold(n_splits=5,  shuffle=True, random_state=42),
}

timing_results = {}
score_results  = {}

for name, cv in cv_configs.items():
    times_run = []
    for _ in range(5):          # average over 5 runs for stable timing
        t0 = time.time()
        sc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
        times_run.append(time.time() - t0)
    timing_results[name] = np.mean(times_run)
    score_results[name]  = sc
    print(f"  {name:<18} | mean acc = {sc.mean():.4f} | time = {np.mean(times_run):.3f}s")

print("\nSpeed ratio (LOOCV / 5-Fold):",
      f"{timing_results['LOOCV (k=150)'] / timing_results['5-Fold CV']:.1f}x slower")

In [ ]:
# ── Cell 7: Variance comparison — 50 repeated runs ────────────────────────────
# LOOCV is deterministic (no randomness in splits) but its ESTIMATE has
# high variance because each fold's test set is just 1 sample.
# We simulate variance by re-running on bootstrap subsamples.

N_REPEATS = 50
cv_variance = {'LOOCV': [], '10-Fold': [], '5-Fold': []}

for rep in range(N_REPEATS):
    # Bootstrap resample so splits differ each repetition
    idx = np.random.choice(N, size=N, replace=True)
    Xr, yr = X[idx], y[idx]

    # LOOCV — deterministic on THIS resample
    sc_loo = cross_val_score(pipe, Xr, yr, cv=LeaveOneOut(), scoring='accuracy')
    cv_variance['LOOCV'].append(sc_loo.mean())

    # 10-fold
    sc_10 = cross_val_score(pipe, Xr, yr,
                            cv=StratifiedKFold(10, shuffle=True, random_state=rep),
                            scoring='accuracy')
    cv_variance['10-Fold'].append(sc_10.mean())

    # 5-fold
    sc_5 = cross_val_score(pipe, Xr, yr,
                           cv=StratifiedKFold(5, shuffle=True, random_state=rep),
                           scoring='accuracy')
    cv_variance['5-Fold'].append(sc_5.mean())

# Report standard deviations
print("Variance of CV score estimate across 50 bootstrap resamples:")
print("-" * 50)
for name, vals in cv_variance.items():
    print(f"  {name:<10} | std = {np.std(vals):.5f}  | mean = {np.mean(vals):.4f}")

In [ ]:
# ── Cell 8: Box plot of score distributions ────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Left: Box plots of repeated CV estimates ----
ax = axes[0]
var_df = pd.DataFrame(cv_variance)
var_df.boxplot(ax=ax, notch=False, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6),
               medianprops=dict(color='red', linewidth=2))
ax.set_title('Distribution of CV Score Estimates\n(50 bootstrap resamples, n=150 fraud dataset)', fontsize=11)
ax.set_ylabel('Accuracy Score')
ax.set_xlabel('CV Strategy')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

# Annotate variance
for i, (name, vals) in enumerate(cv_variance.items(), 1):
    ax.text(i, np.min(vals) - 0.01,
            f'σ={np.std(vals):.4f}', ha='center', va='top', fontsize=9, color='navy')

# ---- Right: Bias-Variance conceptual diagram ----
ax2 = axes[1]
k_vals  = np.array([2, 5, 10, 20, 50, 150])
# Conceptual: bias decreases with k; variance peaks near k=n (LOOCV)
bias_curve     = 1.0 / k_vals + 0.02
variance_curve = 0.01 + 0.001 * k_vals
total_curve    = bias_curve + variance_curve

ax2.plot(k_vals, bias_curve,     'o-', label='Bias² (approx)', color='tomato')
ax2.plot(k_vals, variance_curve, 's-', label='Variance (approx)', color='steelblue')
ax2.plot(k_vals, total_curve,    '^-', label='Total Error', color='purple', linewidth=2)
ax2.axvline(x=5,   linestyle='--', alpha=0.5, color='green',  label='5-fold')
ax2.axvline(x=10,  linestyle='--', alpha=0.5, color='orange', label='10-fold')
ax2.axvline(x=150, linestyle='--', alpha=0.5, color='red',    label='LOOCV (k=n)')
ax2.set_xlabel('Number of Folds (k)')
ax2.set_ylabel('Conceptual Error Magnitude')
ax2.set_title('Bias-Variance Tradeoff of CV Strategies\n(Conceptual — not to scale)', fontsize=11)
ax2.legend(fontsize=8)
ax2.set_xscale('log')

plt.tight_layout()
plt.suptitle('LOOCV: Low Bias, Higher Variance Than k-Fold', y=1.02, fontsize=13, fontweight='bold')
plt.show()

print("Key insight: LOOCV has the lowest BIAS (trains on n-1 ≈ n samples)")
print("but higher VARIANCE because each test set is just 1 sample — unstable estimates.")

## The Analytical Shortcut: Hat Matrix LOOCV

For **OLS (linear) regression**, there is a beautiful algebraic identity that lets you compute the exact LOOCV MSE from a single model fit:

$$\text{LOOCV MSE} = \frac{1}{n} \sum_{i=1}^{n} \left( \frac{y_i - \hat{y}_i}{1 - h_{ii}} \right)^2$$

**Hat matrix** $H = X(X^\top X)^{-1}X^\top$:
- Maps the observed $y$ to the fitted values: $\hat{y} = Hy$
- The diagonal entries $h_{ii}$ measure **leverage** — how unusual sample i's features are
- $0 \leq h_{ii} \leq 1$; high $h_{ii}$ means sample i strongly influences its own prediction

**Why does this work?** The OLS estimator is linear in y. When you remove point i and refit, the change in the fitted value at point i can be expressed exactly in terms of $h_{ii}$ and the full-model residual. The denominator $(1 - h_{ii})$ inflates the residual to account for the fact that you did NOT train on that point.

**Why can't you do this for neural networks?** Neural networks are non-linear functions of their parameters. There is no closed-form expression for how removing one training point affects the learned weights. The hat matrix trick is specific to linear projections.

In [ ]:
# ── Cell 10: Hat matrix LOOCV — analytical computation ───────────────────────
# Task: predict fraud transaction AMOUNT from other features (regression)

# Use a regression target: log(transaction_amount) from remaining features
feature_cols = ['time_of_day', 'merchant_risk', 'txn_today', 'dist_from_home', 'is_fraud']
X_reg = df[feature_cols].values
y_reg = np.log1p(df['transaction_amount'].values)   # log-transform for normality
n_reg = len(y_reg)

# Add intercept column
X_design = np.column_stack([np.ones(n_reg), StandardScaler().fit_transform(X_reg)])

# ── Analytical LOOCV via hat matrix ──────────────────────────────────────────
def hat_matrix_loocv_mse(X_design, y):
    """Compute LOOCV MSE for OLS using the hat matrix shortcut."""
    # Hat matrix H = X (XᵀX)⁻¹ Xᵀ
    XtXinv = np.linalg.inv(X_design.T @ X_design)
    H = X_design @ XtXinv @ X_design.T          # shape: (n, n)
    h_diag = np.diag(H)                          # leverage scores h_ii

    # Full model predictions
    beta = XtXinv @ X_design.T @ y              # OLS coefficients
    y_hat = X_design @ beta                     # fitted values
    residuals = y - y_hat                       # full-model residuals

    # LOOCV MSE via shortcut formula
    loocv_residuals = residuals / (1 - h_diag)   # inflate by leverage
    loocv_mse = np.mean(loocv_residuals ** 2)
    return loocv_mse, h_diag, residuals

analytical_mse, leverages, residuals = hat_matrix_loocv_mse(X_design, y_reg)
print(f"Analytical LOOCV MSE (hat matrix): {analytical_mse:.6f}")

In [ ]:
# ── Cell 11: Verify analytical = iterative LOOCV for linear regression ────────

def iterative_loocv_mse(X_design, y):
    """Brute-force LOOCV: fit n separate OLS models."""
    n = len(y)
    loo_errors = np.zeros(n)
    for i in range(n):
        idx = np.concatenate([np.arange(0, i), np.arange(i+1, n)])
        Xtr, ytr = X_design[idx], y[idx]
        # OLS on training set
        beta_i = np.linalg.lstsq(Xtr, ytr, rcond=None)[0]
        y_pred_i = X_design[i] @ beta_i          # predict on held-out sample
        loo_errors[i] = (y[i] - y_pred_i) ** 2
    return np.mean(loo_errors)

print("Running iterative LOOCV (150 OLS fits)...")
t0 = time.time()
iterative_mse = iterative_loocv_mse(X_design, y_reg)
t_iter = time.time() - t0
print(f"  Iterative LOOCV MSE:  {iterative_mse:.6f}  (took {t_iter:.3f}s)")

t0 = time.time()
analytical_mse, _, _ = hat_matrix_loocv_mse(X_design, y_reg)
t_anal = time.time() - t0
print(f"  Analytical LOOCV MSE: {analytical_mse:.6f}  (took {t_anal:.4f}s)")

print(f"\n  Difference: {abs(iterative_mse - analytical_mse):.2e}  (should be ~0)")
print(f"  Speed-up:   {t_iter/max(t_anal, 1e-6):.0f}x faster with hat matrix")
print("\nConclusion: Analytical and iterative LOOCV are mathematically identical for OLS.")

In [ ]:
# ── Cell 12: Leverage score visualisation ────────────────────────────────────
# High-leverage points are outliers in feature space — they dominate LOOCV

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Left: leverage histogram ----
ax = axes[0]
ax.hist(leverages, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(leverages.mean(), color='red', linestyle='--', label=f'Mean h_ii = {leverages.mean():.3f}')
p_threshold = 2 * X_design.shape[1] / n_reg   # common rule of thumb
ax.axvline(p_threshold, color='orange', linestyle=':', linewidth=2,
           label=f'2p/n threshold = {p_threshold:.3f}')
ax.set_xlabel('Leverage h_ii')
ax.set_ylabel('Count')
ax.set_title('Leverage Scores (Hat Matrix Diagonal)\nHigh leverage = unusual X values', fontsize=11)
ax.legend()

# ---- Right: residual vs leverage (influence plot) ----
ax2 = axes[1]
sc = ax2.scatter(leverages, residuals, c=np.abs(residuals/(1-leverages)),
                 cmap='YlOrRd', edgecolors='grey', s=50, alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axvline(p_threshold, color='orange', linestyle=':', linewidth=2, label='2p/n threshold')
plt.colorbar(sc, ax=ax2, label='LOOCV residual (inflated)')
ax2.set_xlabel('Leverage h_ii')
ax2.set_ylabel('OLS Residual')
ax2.set_title('Influence Plot: Leverage vs Residual\nBright = large LOOCV contribution', fontsize=11)
ax2.legend()

plt.tight_layout()
plt.suptitle('Understanding Leverage in LOOCV', y=1.01, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# ── Cell 13: Computational scaling — O(n) growth of LOOCV ────────────────────
# Demonstrate that LOOCV time scales linearly with n

sample_sizes = [50, 100, 200, 300, 500, 750, 1000]
loocv_times  = []
kfold5_times = []

for n_samples in sample_sizes:
    # Generate a dataset of this size
    Xn, yn = make_classification(
        n_samples=n_samples, n_features=5, n_informative=3,
        weights=[0.95, 0.05], random_state=42
    )
    pipe_n = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=200, random_state=42))
    ])

    # Time LOOCV
    t0 = time.time()
    cross_val_score(pipe_n, Xn, yn, cv=LeaveOneOut(), scoring='accuracy')
    loocv_times.append(time.time() - t0)

    # Time 5-fold
    t0 = time.time()
    cross_val_score(pipe_n, Xn, yn, cv=5, scoring='accuracy')
    kfold5_times.append(time.time() - t0)

    print(f"n={n_samples:5d} | LOOCV={loocv_times[-1]:.3f}s | 5-Fold={kfold5_times[-1]:.4f}s "
          f"| ratio={loocv_times[-1]/kfold5_times[-1]:.1f}x")

In [ ]:
# ── Cell 14: Plot O(n) scaling ────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Left: absolute timing ----
ax = axes[0]
ax.plot(sample_sizes, loocv_times,  'o-', color='tomato',    label='LOOCV',  linewidth=2)
ax.plot(sample_sizes, kfold5_times, 's-', color='steelblue', label='5-Fold', linewidth=2)

# Fit a linear curve through LOOCV points to show O(n) trend
coeffs = np.polyfit(sample_sizes, loocv_times, 1)
fitted_line = np.poly1d(coeffs)(sample_sizes)
ax.plot(sample_sizes, fitted_line, '--', color='red', alpha=0.5, label='O(n) fit')

ax.set_xlabel('Dataset Size (n)')
ax.set_ylabel('Wall-Clock Time (seconds)')
ax.set_title('LOOCV Scales as O(n) — Linear in Dataset Size\n5-Fold is approximately O(1) relative to n', fontsize=10)
ax.legend()

# ---- Right: ratio ----
ax2 = axes[1]
ratios = [l / f for l, f in zip(loocv_times, kfold5_times)]
ax2.bar(sample_sizes, ratios, color='orchid', edgecolor='white', width=40)
ax2.set_xlabel('Dataset Size (n)')
ax2.set_ylabel('LOOCV / 5-Fold Time Ratio')
ax2.set_title('How Much Slower Is LOOCV?\n(ratio grows with n)', fontsize=11)
for x, r in zip(sample_sizes, ratios):
    ax2.text(x, r + 0.3, f'{r:.0f}x', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.suptitle('Computational Cost: LOOCV vs 5-Fold', y=1.01, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# ── Cell 15: sklearn LOOCV — full worked example with fraud data ──────────────
# Comparing three classifiers using LOOCV on the small fraud dataset

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

classifiers = {
    'Logistic Regression (stable)':  LogisticRegression(max_iter=300, random_state=42),
    'k-NN k=1 (very unstable)':      KNeighborsClassifier(n_neighbors=1),
    'Decision Tree max_depth=None':  DecisionTreeClassifier(random_state=42),
}

loo = LeaveOneOut()

print(f"{'Model':<35} | {'LOOCV Acc':>10} | {'5-Fold Acc':>10} | {'Diff':>8}")
print("-" * 72)

for name, clf in classifiers.items():
    pipe_clf = Pipeline([('sc', StandardScaler()), ('clf', clf)])

    sc_loo = cross_val_score(pipe_clf, X, y, cv=loo, scoring='accuracy')
    sc_5   = cross_val_score(pipe_clf, X, y,
                             cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             scoring='accuracy')

    diff = sc_loo.mean() - sc_5.mean()
    print(f"{name:<35} | {sc_loo.mean():>10.4f} | {sc_5.mean():>10.4f} | {diff:>+8.4f}")

print("\nNote: k-NN k=1 and deep trees are UNSTABLE — small changes in training")
print("set cause large prediction swings. LOOCV score variance is highest for these.")

## When to Use LOOCV — Decision Framework

Use the table below to decide your cross-validation strategy:

| Dataset Size | Model Stability | Recommendation | Reason |
|:---:|:---:|:---:|:---|
| n < 50 | Stable (linear, regularised) | **LOOCV** | Data too precious to waste on large val sets |
| n < 50 | Unstable (deep tree, k-NN k=1) | **LOOCV + caution** | High variance; consider regularising the model |
| 50 ≤ n ≤ 200 | Stable | **10-fold** | Good bias-variance balance, affordable cost |
| 50 ≤ n ≤ 200 | Unstable | **10-fold repeated** | Repeat 3x to reduce variance |
| n > 1000 | Any | **5-fold** | LOOCV costs n fits — too slow |
| Time series | Any | **TimeSeriesSplit** | LOOCV ignores temporal order → leakage |
| Severe class imbalance | Any | **StratifiedKFold** | Ensures each fold has minority class |
| Linear regression | Any | **Hat matrix LOOCV** | Analytical shortcut — exact and O(1) |

---

## The Paradox: More Data Per Fold → More Correlated Folds → Higher Variance

This is the most counter-intuitive fact about LOOCV:

- **LOOCV** trains on n-1 samples per fold (nearly all data) → lowest bias
- But the n folds overlap almost entirely — they share n-2 training samples
- Highly overlapping folds produce **highly correlated** predictions
- Averaging correlated variables does not reduce variance as much as averaging independent variables
- Result: the LOOCV estimate has **higher variance** than 5-fold or 10-fold, even though each individual prediction is made by a model trained on more data

This is NOT the same as model variance. It is the **variance of the evaluation metric** — how much the score would change if you re-ran the experiment.

In [ ]:
# ── Cell 17: Visualise fold overlap — why LOOCV folds are correlated ──────────

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
n_show = 20   # show first 20 samples for clarity

configs = [
    ('LOOCV (k=20)', LeaveOneOut(),     'tomato'),
    ('10-Fold CV',   KFold(n_splits=10), 'steelblue'),
    ('5-Fold CV',    KFold(n_splits=5),  'seagreen'),
]

X_small = X[:n_show]
y_small = y[:n_show]

for ax, (title, cv_obj, color) in zip(axes, configs):
    fold_matrix = np.zeros((n_show, n_show), dtype=int)
    fold_idx    = np.zeros(n_show, dtype=int)

    for fold_num, (train_idx, test_idx) in enumerate(cv_obj.split(X_small, y_small)):
        fold_idx[test_idx] = fold_num

    # Show which fold each sample belongs to
    fold_labels = fold_idx.reshape(1, -1)
    im = ax.imshow(fold_labels, cmap='tab10', aspect='auto',
                   vmin=0, vmax=max(fold_idx))

    # Annotate
    for j in range(n_show):
        ax.text(j, 0, str(fold_idx[j]), ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')

    ax.set_xlim(-0.5, n_show - 0.5)
    ax.set_yticks([])
    ax.set_xticks(range(0, n_show, 5))
    ax.set_xlabel('Sample Index')
    ax.set_title(f'{title}\nEach number = fold assignment', fontsize=10)

plt.suptitle('Fold Assignments for First 20 Samples\nLOOCV: each sample is its own fold → maximum overlap between training sets',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 18: Recommendation table — programmatic output ──────────────────────

recommendations = pd.DataFrame([
    {'Dataset Size':   'n < 50',
     'Model Stability': 'Stable',
     'CV Strategy':    'LOOCV',
     'sklearn API':    'LeaveOneOut()',
     'Key Risk':       'Slow if model is complex'},
    {'Dataset Size':   'n < 50',
     'Model Stability': 'Unstable',
     'CV Strategy':    'LOOCV + regularise model',
     'sklearn API':    'LeaveOneOut()',
     'Key Risk':       'High variance estimate'},
    {'Dataset Size':   '50–200',
     'Model Stability': 'Either',
     'CV Strategy':    '10-Fold',
     'sklearn API':    'KFold(10)',
     'Key Risk':       'Mild bias'},
    {'Dataset Size':   '200–1000',
     'Model Stability': 'Either',
     'CV Strategy':    '5-Fold or 10-Fold',
     'sklearn API':    'KFold(5) or KFold(10)',
     'Key Risk':       'Acceptable bias, fast'},
    {'Dataset Size':   'n > 1000',
     'Model Stability': 'Either',
     'CV Strategy':    '5-Fold',
     'sklearn API':    'KFold(5)',
     'Key Risk':       'LOOCV too slow'},
    {'Dataset Size':   'Any (time series)',
     'Model Stability': 'Either',
     'CV Strategy':    'TimeSeriesSplit',
     'sklearn API':    'TimeSeriesSplit()',
     'Key Risk':       'Standard CV leaks future data'},
    {'Dataset Size':   'Any (OLS regression)',
     'Model Stability': 'Stable (linear)',
     'CV Strategy':    'Hat matrix LOOCV',
     'sklearn API':    'Manual: H = X(XᵀX)⁻¹Xᵀ',
     'Key Risk':       'Only valid for OLS'},
])

pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 120)
print("CV Strategy Recommendation Table")
print("=" * 110)
print(recommendations.to_string(index=False))

In [ ]:
# ── Cell 19: Full sklearn example — LOOCV with cross_val_score ───────────────
# Best practice: use sklearn's built-in LOOCV for production code

from sklearn.model_selection import cross_validate

pipe_final = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=0.1, max_iter=300, random_state=42))
])

# cross_val_score with LeaveOneOut() — simplest API
loo_scores = cross_val_score(
    pipe_final, X, y,
    cv=LeaveOneOut(),
    scoring='accuracy',
    n_jobs=-1           # parallelise across CPU cores
)

# cross_validate gives additional detail
loo_detailed = cross_validate(
    pipe_final, X, y,
    cv=LeaveOneOut(),
    scoring=['accuracy', 'roc_auc'],
    n_jobs=-1
)

print("LOOCV Results (sklearn LeaveOneOut):")
print(f"  n folds evaluated: {len(loo_scores)}")
print(f"  Mean accuracy:     {loo_scores.mean():.4f}")
print(f"  Std of scores:     {loo_scores.std():.4f}")
print(f"  Min fold score:    {loo_scores.min():.1f}  (a fold that got it wrong)")
print(f"  Max fold score:    {loo_scores.max():.1f}  (a fold that got it right)")
print("\nNote: In LOOCV, each fold score is either 0.0 or 1.0 for classification.")
print("The mean is the accuracy; std measures how often the model makes mistakes.")

In [ ]:
# ── Cell 20: Summary visualisation — full LOOCV picture ──────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ---- Top-left: per-fold scores (0 or 1) ----
ax = axes[0, 0]
colors = ['tomato' if s == 0 else 'steelblue' for s in loo_scores]
ax.bar(range(len(loo_scores)), loo_scores, color=colors, width=1.0)
ax.axhline(loo_scores.mean(), color='black', linestyle='--',
           label=f'Mean = {loo_scores.mean():.3f}')
ax.set_xlabel('Sample Index (fold i)')
ax.set_ylabel('Correct Prediction (1=yes, 0=no)')
ax.set_title(f'LOOCV: Per-Fold Prediction Accuracy\n'
             f'Red = missed, Blue = correct | n={len(loo_scores)} folds')
ax.legend()

# ---- Top-right: error distribution ----
ax2 = axes[0, 1]
wrong_idx = np.where(loo_scores == 0)[0]
wrong_labels = y[wrong_idx]
ax2.bar(['Missed Fraud\n(FN)', 'Missed Legit\n(FP)'],
        [(wrong_labels == 1).sum(), (wrong_labels == 0).sum()],
        color=['tomato', 'orange'])
ax2.set_title('What Type of Mistakes Did LOOCV Expose?')
ax2.set_ylabel('Number of Misclassifications')
for i, v in enumerate([(wrong_labels == 1).sum(), (wrong_labels == 0).sum()]):
    ax2.text(i, v + 0.05, str(v), ha='center', fontweight='bold')

# ---- Bottom-left: box plot comparison across CV strategies ----
ax3 = axes[1, 0]
var_df.boxplot(ax=ax3, patch_artist=True,
               boxprops=dict(facecolor='lightblue', alpha=0.7),
               medianprops=dict(color='red', linewidth=2))
ax3.set_title('Score Distribution Across\n50 Bootstrap Resamples')
ax3.set_ylabel('CV Accuracy Estimate')
ax3.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

# ---- Bottom-right: LOOCV vs iterative confirmation ----
ax4 = axes[1, 1]
summary_data = {
    'Method': ['LOOCV (n=150 fits)', '5-Fold (5 fits)', '10-Fold (10 fits)'],
    'Accuracy': [
        score_results['LOOCV (k=150)'].mean(),
        score_results['5-Fold CV'].mean(),
        score_results['10-Fold CV'].mean(),
    ],
    'Std': [
        score_results['LOOCV (k=150)'].std(),
        score_results['5-Fold CV'].std(),
        score_results['10-Fold CV'].std(),
    ]
}
sdf = pd.DataFrame(summary_data)
bars = ax4.barh(sdf['Method'], sdf['Accuracy'],
                xerr=sdf['Std'], color=['tomato', 'steelblue', 'seagreen'],
                alpha=0.8, capsize=5)
ax4.set_xlabel('Mean CV Accuracy')
ax4.set_title('CV Strategy Comparison\n(error bars = std of fold scores)')
ax4.set_xlim(0.85, 1.01)

plt.tight_layout()
plt.suptitle('LOOCV — Complete Summary Dashboard', y=1.01, fontsize=14, fontweight='bold')
plt.show()

## Summary: Key Takeaways

| Property | LOOCV | 10-Fold | 5-Fold |
|:---|:---:|:---:|:---:|
| Number of model fits | n | 10 | 5 |
| Training set size per fold | n-1 | ~0.9n | ~0.8n |
| Bias of estimate | Lowest | Low | Moderate |
| Variance of estimate | Highest | Moderate | Lowest |
| Deterministic? | Yes | No (shuffle) | No (shuffle) |
| Computational cost | O(n) fits | O(1) fits | O(1) fits |
| Analytical shortcut? | OLS only | No | No |

**The Golden Rule:** Use LOOCV when n < 50 or when you have a stable linear model and can afford the cost. For everything else, 5-fold or 10-fold with stratification is the workhorse choice.

---

## Self-Check Questions

Test your understanding before moving on. Write your answer, then reveal.

---

**Question 1.** You have n=30 confirmed fraud cases to train a model. Why might LOOCV be better than 5-fold CV here?

<details>
<summary>Reveal Answer</summary>

With n=30 and 5-fold CV, each training fold uses only 24 samples (0.8 × 30) and each validation fold has 6 samples. With so few fraud cases (~5% rate means roughly 1–2 fraud samples per fold), the validation sets will frequently contain no fraud — making the score unreliable.

LOOCV trains on 29 samples each time, keeping almost all signal in the model. Each test set is exactly 1 sample — deterministic, no randomness in splits. The LOOCV estimate, while higher variance, is based on models that each saw nearly all the data, so it is less biased. At n=30, the computational cost (30 fits) is also trivial.

In practice: use `StratifiedKFold` or LOOCV, and consider synthetic oversampling (SMOTE) to balance classes.

</details>

---

**Question 2.** LOOCV has near-zero bias but high variance. How is this different from a high-variance MODEL?

<details>
<summary>Reveal Answer</summary>

These are two different kinds of variance:

- **High-variance MODEL** (e.g., deep decision tree): the model's *predictions* change dramatically when you retrain it on a slightly different dataset. The model is overfitting — it memorises noise. This is a property of the model's predictions on unseen data.

- **High-variance LOOCV ESTIMATE**: the *performance score* you compute using LOOCV changes significantly depending on which dataset you happened to collect. This is a property of the evaluation procedure, not the model. Even a very stable model (like regularised logistic regression) will show high-variance LOOCV scores if n is small, because the 1-sample test sets produce noisy score estimates.

Analogy: a thermometer with high variance gives different readings each time you measure the same temperature (high-variance estimator). A person who is always cold or always hot no matter the temperature is a high-variance model.

</details>

---

**Question 3.** For OLS regression, you can compute LOOCV without refitting n times. Why can't you do this for a neural network?

<details>
<summary>Reveal Answer</summary>

The hat matrix shortcut works because OLS is a **linear** function of the training labels y. The OLS estimator is:

$$\hat{y} = H y \quad \text{where} \quad H = X(X^\top X)^{-1}X^\top$$

Because H is a fixed projection matrix (depends only on X, not y), you can derive the exact change in prediction when you remove point i. The denominator $(1-h_{ii})$ is the closed-form correction.

A neural network's predictions are a **non-linear** function of its parameters, and its parameters are found by iterative gradient descent — not a closed-form formula. There is no equivalent of H for a neural network. Removing one training point changes the loss landscape, the gradient path, possibly the stopping point — in ways that cannot be expressed analytically. You must actually retrain the network n times.

Some approximate influence function methods exist for neural networks (see Koh & Liang 2017), but they are approximations, not exact LOOCV.

</details>